<a href="https://colab.research.google.com/github/NamishBansal15/substation-detection/blob/main/inference-mapping/count-generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/shapefiles/"))

['us_states.shp', 'states.zip', 'cb_2018_us_state_500k.shp.ea.iso.xml', 'cb_2018_us_state_500k.shp.iso.xml', 'cb_2018_us_state_500k.shp', 'cb_2018_us_state_500k.shx', 'cb_2018_us_state_500k.dbf', 'cb_2018_us_state_500k.prj', 'cb_2018_us_state_500k.cpg']


In [ ]:
# =========================
# STEP 0 — INSTALLS
# =========================
!pip install geopandas shapely fiona pyproj rtree

# =========================
# STEP 1 — IMPORTS
# =========================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os
import zipfile

# =========================
# STEP 2 — PATHS
# =========================
BASE_DIR = "/content/drive/MyDrive/Substation_Project/dataset-inference/data/"

PRED_PATH = BASE_DIR + "component_predictions.csv"
META_PATH = BASE_DIR + "image_metadata_bigdataset.csv"

OUTPUT_STATE = BASE_DIR + "results/state_totals.csv"
OUTPUT_NERC = BASE_DIR + "results/nerc_totals.csv"

SHAPE_DIR = "/content/drive/MyDrive/shapefiles/"
ZIP_PATH = SHAPE_DIR + "states.zip"
STATES_PATH = "/content/drive/MyDrive/shapefiles/cb_2018_us_state_500k.shp"

os.makedirs(BASE_DIR + "results/", exist_ok=True)

# =========================
# STEP 3 — LOAD DATA
# =========================
preds = pd.read_csv(PRED_PATH)
meta = pd.read_csv(META_PATH)

# Fix column name
preds = preds.rename(columns={"image": "image_path"})

print("Preds columns:", preds.columns)
print("Meta columns:", meta.columns)

# =========================
# STEP 4 — CLEAN PATHS
# =========================
preds["image_path"] = preds["image_path"].apply(lambda x: x.split("/")[-1])
meta["image_path"] = meta["image_path"].apply(lambda x: x.split("/")[-1])

# =========================
# STEP 5 — MERGE (use ID — safer)
# =========================
df = preds.merge(meta, on="id", how="left")

print("Merged shape:", df.shape)
print("Missing coords:", df["latitude"].isna().sum())

# =========================
# STEP 6 — GEO DATA
# =========================
geometry = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# =========================
# STEP 7 — EXTRACT SHAPEFILE
# =========================
if not os.path.exists(STATES_PATH):
    print("Extracting shapefile...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(SHAPE_DIR)

print("Files in shapefile dir:", os.listdir(SHAPE_DIR))

# =========================
# STEP 8 — LOAD STATES
# =========================
states = gpd.read_file(STATES_PATH).to_crs("EPSG:4326")

print("States columns:", states.columns)

# =========================
# STEP 9 — SPATIAL JOIN
# =========================
gdf = gpd.sjoin(gdf, states, how="left", predicate="within")

# Assign state column
if "STUSPS" in gdf.columns:
    gdf["STATE"] = gdf["STUSPS"]
elif "NAME" in gdf.columns:
    gdf["STATE"] = gdf["NAME"]
else:
    raise ValueError("State column not found in shapefile")

# =========================
# STEP 10 — STATE → NERC
# =========================
state_to_nerc = {
    "TX": "ERCOT",

    "CA": "WECC", "WA": "WECC", "OR": "WECC", "NV": "WECC",
    "ID": "WECC", "UT": "WECC", "AZ": "WECC", "MT": "WECC",
    "WY": "WECC", "CO": "WECC", "NM": "WECC",

    "MN": "MRO", "ND": "MRO", "SD": "MRO", "IA": "MRO", "WI": "MRO",

    "KS": "SPP", "OK": "SPP", "NE": "SPP",

    "IL": "RFC", "IN": "RFC", "OH": "RFC", "MI": "RFC",
    "PA": "RFC", "NJ": "RFC", "MD": "RFC", "DE": "RFC",
    "WV": "RFC", "VA": "RFC",

    "AL": "SERC", "GA": "SERC", "FL": "SERC",
    "MS": "SERC", "NC": "SERC", "SC": "SERC",
    "TN": "SERC", "KY": "SERC",

    "NY": "NPCC", "VT": "NPCC", "NH": "NPCC",
    "ME": "NPCC", "MA": "NPCC", "CT": "NPCC", "RI": "NPCC",
}

gdf["NERC"] = gdf["STATE"].map(state_to_nerc)

# =========================
# STEP 11 — CLEAN
# =========================
gdf = gdf.dropna(subset=["STATE", "NERC"])

print("Final rows:", len(gdf))

# =========================
# STEP 12 — AGGREGATE
# =========================
numeric_cols = gdf.select_dtypes(include="number").columns

state_totals = gdf.groupby("STATE")[numeric_cols].sum()
nerc_totals = gdf.groupby("NERC")[numeric_cols].sum()

# =========================
# STEP 13 — SAVE
# =========================
state_totals.to_csv(OUTPUT_STATE)
nerc_totals.to_csv(OUTPUT_NERC)

print("✅ Done!")
print("State totals:", OUTPUT_STATE)
print("NERC totals:", OUTPUT_NERC)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Substation_Project/data/component_predictions.csv'

In [ ]:
print("Pred columns:", preds.columns)
print("Meta columns:", meta.columns)

Pred columns: Index(['id', 'image_path', 'Circuit Breaker', 'Transformer', 'Reactor',
       'Alt Energy', 'Control', 'Power Lines'],
      dtype='object')
Meta columns: Index(['id', 'latitude', 'longitude', 'region', 'image_path', 'resolution'], dtype='object')


In [ ]:
# =========================
# STEP 0 — INSTALLS
# =========================
!pip install geopandas shapely fiona pyproj rtree


# =========================
# STEP 1 — IMPORTS
# =========================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os
import zipfile


# =========================
# STEP 2 — PATHS
# =========================
BASE_DIR = "/content/drive/MyDrive/dataset-inference/"

PRED_PATH = BASE_DIR + "data/component_predictions.csv"
META_PATH = BASE_DIR + "data/image_metadata.csv"

OUTPUT_STATE = BASE_DIR + "results/state_totals.csv"
OUTPUT_NERC  = BASE_DIR + "results/nerc_totals.csv"

SHAPE_DIR   = "/content/drive/MyDrive/shapefiles/"
ZIP_PATH    = SHAPE_DIR + "states.zip"
STATES_PATH = SHAPE_DIR + "cb_2018_us_state_500k.shp"

# ✅ NEW: NERC GEOJSON
NERC_PATH = BASE_DIR + "nerc_gdf.geojson"

os.makedirs(BASE_DIR + "results/", exist_ok=True)


# =========================
# STEP 3 — LOAD DATA
# =========================
preds = pd.read_csv(PRED_PATH)
meta  = pd.read_csv(META_PATH)

preds = preds.rename(columns={"image": "image_path"})


# =========================
# STEP 4 — CLEAN PATHS
# =========================
preds["image_path"] = preds["image_path"].str.split("/").str[-1]
meta["image_path"]  = meta["image_path"].str.split("/").str[-1]


# =========================
# STEP 5 — MERGE (ROBUST)
# =========================
if "id" in preds.columns and "id" in meta.columns:
    df = preds.merge(meta, on="id", how="left")
else:
    df = preds.merge(meta, on="image_path", how="left")

print("Merged shape:", df.shape)


# =========================
# STEP 6 — CLEAN COORDS
# =========================
df["latitude"]  = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

df = df.dropna(subset=["latitude", "longitude"])

print("Valid rows:", len(df))


# =========================
# STEP 7 — GEO DATAFRAME
# =========================
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)


# =========================
# STEP 8 — STATES SHAPEFILE
# =========================
if not os.path.exists(STATES_PATH):
    print("Extracting shapefile...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(SHAPE_DIR)

states = gpd.read_file(STATES_PATH).to_crs("EPSG:4326")


# =========================
# STEP 9 — STATE JOIN
# =========================
gdf = gpd.sjoin(gdf, states, how="left", predicate="within")

if "STUSPS" in gdf.columns:
    gdf["STATE"] = gdf["STUSPS"]
elif "NAME" in gdf.columns:
    gdf["STATE"] = gdf["NAME"]
else:
    raise ValueError("State column not found")


# =========================
# STEP 10 — NERC JOIN (NEW ✅)
# =========================
if os.path.exists(NERC_PATH):
    print("Using NERC GeoJSON (better than mapping)...")

    gdf_nerc = gpd.read_file(NERC_PATH).to_crs("EPSG:4326")

    # detect region column
    possible_cols = ["NERC", "region", "NAME"]
    region_col = next((c for c in possible_cols if c in gdf_nerc.columns), None)

    if region_col is None:
        raise ValueError(f"NERC column not found: {gdf_nerc.columns}")

    gdf = gpd.sjoin(gdf, gdf_nerc[[region_col, "geometry"]], how="left", predicate="intersects")

    gdf["NERC"] = gdf[region_col]

else:
    print("Fallback to state → NERC mapping")

    state_to_nerc = {
        "TX": "ERCOT",
        "CA": "WECC","WA": "WECC","OR": "WECC","NV": "WECC",
        "ID": "WECC","UT": "WECC","AZ": "WECC","MT": "WECC",
        "WY": "WECC","CO": "WECC","NM": "WECC",
        "MN": "MRO","ND": "MRO","SD": "MRO","IA": "MRO","WI": "MRO",
        "KS": "SPP","OK": "SPP","NE": "SPP",
        "IL": "RFC","IN": "RFC","OH": "RFC","MI": "RFC",
        "PA": "RFC","NJ": "RFC","MD": "RFC","DE": "RFC",
        "WV": "RFC","VA": "RFC",
        "AL": "SERC","GA": "SERC","FL": "SERC",
        "MS": "SERC","NC": "SERC","SC": "SERC",
        "TN": "SERC","KY": "SERC",
        "NY": "NPCC","VT": "NPCC","NH": "NPCC",
        "ME": "NPCC","MA": "NPCC","CT": "NPCC","RI": "NPCC",
    }

    gdf["NERC"] = gdf["STATE"].map(state_to_nerc)


# =========================
# STEP 11 — CLEAN
# =========================
gdf = gdf.dropna(subset=["STATE", "NERC"])
print("Final rows:", len(gdf))


# =========================
# STEP 12 — SMART AGGREGATION ✅
# =========================
# Only aggregate prediction-related columns
exclude_cols = ["id", "latitude", "longitude"]
numeric_cols = [
    c for c in gdf.select_dtypes(include="number").columns
    if c not in exclude_cols
]

print("Aggregating columns:", numeric_cols)

state_totals = gdf.groupby("STATE")[numeric_cols].sum()
nerc_totals  = gdf.groupby("NERC")[numeric_cols].sum()


# =========================
# STEP 13 — SAVE
# =========================
state_totals.to_csv(OUTPUT_STATE)
nerc_totals.to_csv(OUTPUT_NERC)

print("✅ Done!")
print("State totals:", OUTPUT_STATE)
print("NERC totals:", OUTPUT_NERC)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.4 MB/s eta 0:00:00
Merged shape: (11085, 12)
Valid rows: 11085
Fallback to state → NERC mapping
Final rows: 9920
Aggregating columns: ['Transformer', 'Reactor', 'Circuit Breaker', 'Alt Energy', 'Control', 'Power Lines', 'resolution', 'index_right', 'ALAND', 'AWATER']
✅ Done!
State totals: /content/drive/MyDrive/dataset-inference/results/state_totals.csv
NERC totals: /content/drive/MyDrive/dataset-inference/results/nerc_totals.csv


In [ ]:
BASE_DIR = "/content/drive/MyDrive/dataset-inference"

# search for file automatically
for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if "component_predictions" in file:
            print(os.path.join(root, file))